This is the notebook to run inference on trained model using the `CoppeliaSim` robot simulator. Follow the below steps for running this notebook

1. Download and install CoppeliaSim from here: [https://www.coppeliarobotics.com/](https://www.coppeliarobotics.com/)
2. Open the scene file `\simulation\sim_envs\coppeliasim_scene_for_spraying_v3.ttt` in the simulator software (must do it before running the notebook).
3. Run all the cells in the notebook (there is an assertion error just to prevent running the notebook before opening the scene file when using the run all option. Please continue running below cells if you get assertion error.)
4. For every environment, modify the path for trained model and run the cell.
4. **IMPORTANT**: When closing the CoppeliaSim scene file, please select "No" for the question "Do you wish to save the changes?"

### Load modules and configurations

In [ ]:
import os
import json
import numpy as np
import gymnasium as gym
import pygame

from sb3_contrib import CrossQ
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

In [ ]:
# ================================
# Utility Functions (Optimized)
# ================================
def is_inside_polygon(point, poly):  # check if a point is inside polygon
    x, y = point  # unpack point
    inside = False  # initialize flag
    n = len(poly)  # number of vertices
    p1x, p1y = poly[0]  # first vertex
    for i in range(n + 1):  # iterate through edges
        p2x, p2y = poly[i % n]  # next vertex (wrap around)
        # check if point is within vertical bounds of edge
        if min(p1y, p2y) < y <= max(p1y, p2y) and x <= max(p1x, p2x):
            if p1y != p2y:  # avoid division by zero
                xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x  # intersection
            if p1x == p2x or x <= xinters:  # crossing check
                inside = not inside  # toggle
        p1x, p1y = p2x, p2y  # move to next edge
    return inside  # return result

def compute_min_dist(x):  # compute minimum pairwise distance
    x = np.asarray(x, dtype=np.float32)  # ensure array
    diff = x[:, None, :] - x[None, :, :]  # pairwise differences
    dist_matrix = np.linalg.norm(diff, axis=-1)  # compute distances
    np.fill_diagonal(dist_matrix, np.inf)  # ignore self-distance
    return float(np.min(dist_matrix))  # return min distance

def load_experiment_dict_json(json_path):  # load experiment configs
    with open(json_path, "r") as f:  # open file
        data = json.load(f)  # load JSON
    for _, cfg in data.items():  # iterate over configs
        cfg["field"] = [tuple(p) for p in cfg["field"]]  # convert to tuples
        cfg["init_positions"] = np.array(cfg["init_positions"], dtype=float)  # positions array
        cfg["infected_locations"] = [tuple(p) for p in cfg["infected_locations"]]  # infection
    return data  # return processed dict

# ================================
# Environment
# ================================
class MultiRobotEnv(gym.Env):
    metadata = {'render_modes': ['human', 'print', 'rgb_array'], "render_fps": 60}
    def __init__(
        self,
        field_info,
        render_mode=None,
        wind_par=[0, 0],
        num_robots=3,
        render_scale=5,
        max_steps=1000,
        # ── experiment control ──────────────────────────────────────
        reward_ablation="full",
        obs_mode="full",
        uncertainty_mode="full",
        dr_mode="none",
    ):
        super().__init__()

        # Validate experiment parameters
        assert reward_ablation in ("full", "no_term" , "no_spr" , "no_path"), \
            f"Unknown reward_ablation: {reward_ablation}"
        assert obs_mode in ("full" , "no_pos" , "no_inf_hist" , "pos_only"), \
            f"Unknown obs_mode: {obs_mode}"
        assert uncertainty_mode in ("full" , "wind_only" , "act_only" , "deterministic"), \
            f"Unknown uncertainty_mode: {uncertainty_mode}"
        assert dr_mode in ("none", "wind", "full"), \
            f"Unknown dr_mode: {dr_mode}"
        assert render_mode is None or render_mode in self.metadata["render_modes"]

        # Store experiment flags
        self.reward_ablation  = reward_ablation
        self.obs_mode         = obs_mode
        self.uncertainty_mode = uncertainty_mode
        self.dr_mode          = dr_mode

        # ── Field info ──────────────────────────────────────────────
        self.field_info = field_info
        self.poly_vertices = field_info['field']
        self.xs, self.ys = zip(*self.poly_vertices)
        self.min_x, self.max_x = np.min(self.xs), np.max(self.xs)
        self.min_y, self.max_y = np.min(self.ys), np.max(self.ys)
        self.world_width  = self.max_x - self.min_x
        self.world_height = self.max_y - self.min_y

        # ── Rendering ───────────────────────────────────────────────
        self.render_scale = render_scale
        self.screen_width, self.screen_height = 800, 800
        self.offset_x = (self.screen_width  - self.world_width  * self.render_scale) / 2
        self.offset_y = (self.screen_height - self.world_height * self.render_scale) / 2
        self.max_steps = max_steps        
        self.render_mode = render_mode
        self.screen = None
        self.clock  = None

        # ── Robot params ────────────────────────────────────────────
        self.num_robots = num_robots
        self.init_robot_positions = field_info['init_positions'][:num_robots]
        self.init_robot_capacities = np.array(
            field_info['robot_capacities'][:num_robots], dtype=np.float32)
        self.robot_size   = 2    # collision radius
        self.mass         = 1.0  # UAV mass (overridden by DR full)
        self.thrust_power = 0.5  # action scaling (overridden by DR full)
        self.max_speed    = 5
        self.min_speed    = -5
        self.robot_colors = [
            (255, 0, 0),    # Red
            (0, 255, 0),    # Lime / Green
            (255, 0, 255),  # Magenta / Fuchsia
            (255, 128, 0),  # Orange
            (128, 0, 255),  # Violet / Purple
            (255, 0, 255)   # Magenta / Fuchsia
        ]

        # ── Infection params ────────────────────────────────────────
        self._nominal_infected_radius = 5   # nominal spray radius (DR may override per-episode)
        self.infected_radius = self._nominal_infected_radius
        self.spray_sigma     = self.infected_radius / 2
        self.init_infected_positions = np.array(
            [[x, y] for x, y, _ in field_info['infected_locations']])
        self.init_infected_levels = np.array(
            [lvl for _, _, lvl in field_info['infected_locations']], dtype=np.float32)
        self.max_infection_level = np.max(self.init_infected_levels)

        # ── Base wind (mean) magnitude and direction
        self.base_wind_mag, self.base_wind_dir = wind_par

        # ── Noise stds — set by uncertainty_mode ────────────────────
        # These are the *nominal* values; DR "full" may override
        # action_noise_std at the start of each episode.
        _noise = {
            "full":          dict(wind=0.20, wind_dir=5.0, action=0.10, spray=0.05, obs=0.01),
            "wind_only":     dict(wind=0.20, wind_dir=5.0, action=0.00, spray=0.00, obs=0.00),
            "act_only":      dict(wind=0.00, wind_dir=0.0, action=0.10, spray=0.00, obs=0.00),
            "deterministic": dict(wind=0.00, wind_dir=0.0, action=0.00, spray=0.00, obs=0.00),
        }[uncertainty_mode]

        self.wind_noise_std         = _noise["wind"]
        self.wind_dir_noise_std     = _noise["wind_dir"]
        self.action_noise_std       = _noise["action"]
        self.spray_noise_std        = _noise["spray"]
        self.obs_noise_std          = _noise["obs"]
        self.init_position_noise    = 0.5

        # ── Nominal noise stds (DR "full" samples around these) ─────
        self._nominal_action_noise_std = _noise["action"]
        self._nominal_mass             = 1.0
        self._nominal_thrust_power     = 0.5

        # Each robot's action have the following:
        #   1. a_x = Force component (or thrust) along x-axis
        #   2. a_y = Force component (or thrust) along y-axis
        #   3. sigma = spray rate
        # Therefore, total actions = 3*num_robots

        # ── Action space: (ax, ay, spray) per robot ─────────────────
        self.action_space = gym.spaces.Box(
            low=-1.0, high=1.0, shape=(num_robots, 3), dtype=np.float32)

        # Each robot's observation have the following for base:
        #    1. (x, y) co-ordinates of each robot
        #    2. (v_x, v_y) velocities along x and y axis of each robot
        #    3. current spraying capacity (C_p(t))
        #    4. current infection levels of M locations
        # Therefore, total observations = 2*num_robots (positions) + 2*num_robots (velocities) + 
        # num_robots (spraying capacities) + M (num_inf_locations) = 5*num_robots + M

        # ── Observation space — depends on obs_mode ──────────────────
        M = len(self.init_infected_levels)
        N = num_robots
        _obs_dims = {
            "full":          5*N + M,       # original
            "no_pos":        N + M,         # capacities(N) + inf_levels(M)
            "no_inf_hist":   5*N,     # 5N 
            "pos_only":      2*N,           # positions only
        }
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(_obs_dims[obs_mode],), dtype=np.float32)

        self.reset()  # initialise state

    # ── coordinate conversion ────────────────────────────────────────
    def world_to_screen(self, pos):
        x = (pos[0] - self.min_x) * self.render_scale + self.offset_x
        y = (pos[1] - self.min_y) * self.render_scale + self.offset_y
        return int(x), int(y)

    # ── observation builder ──────────────────────────────────────────
    def _get_obs(self):
        info = {
            f'robot{i}': {
                'position': self.robot_positions[i],
                'capacity': self.robot_capacities[i],
            }
            for i in range(self.num_robots)
        }
        infection_norm = self.infected_levels / self.max_infection_level

        if self.obs_mode == "full":
            state = np.concatenate([self.robot_positions.flatten(),
            self.robot_velocities.flatten(),
            self.robot_capacities,
            infection_norm])
        elif self.obs_mode == "no_pos":
            state = np.concatenate([self.robot_capacities, infection_norm])
        elif self.obs_mode == "no_inf_hist":
            state = np.concatenate([self.robot_positions.flatten(),
            self.robot_velocities.flatten(),
            self.robot_capacities])
        elif self.obs_mode == "pos_only":
            state = self.robot_positions.flatten().copy()

        if self.obs_noise_std > 0:
            state = state + np.random.normal(0, self.obs_noise_std, size=state.shape)

        return state.astype(np.float32), info

    # ── reset ────────────────────────────────────────────────────────
    def reset(self, seed=None, options={}):
        super().reset(seed=seed)

        # ── Domain randomization — re-sample base params each episode ─
        if self.dr_mode == "none":
            # Standard: add small episode-level noise to the nominal wind
            self.wind_mag = self.base_wind_mag + np.random.normal(0, self.wind_noise_std)
            self.wind_dir = self.base_wind_dir + np.random.normal(0, self.wind_dir_noise_std)
            # Restore nominal physical params (may have been overridden last episode)
            self.action_noise_std = self._nominal_action_noise_std
            self.infected_radius  = self._nominal_infected_radius
            self.spray_sigma      = self.infected_radius / 2
            self.mass             = self._nominal_mass
            self.thrust_power     = self._nominal_thrust_power

        elif self.dr_mode == "wind":
            # Randomise wind speed and direction uniformly
            self.wind_mag = float(np.random.uniform(0.0, 1.0))
            self.wind_dir = float(np.degrees(np.random.uniform(0.0, 2 * np.pi)))
            self.action_noise_std = self._nominal_action_noise_std
            self.infected_radius  = self._nominal_infected_radius
            self.spray_sigma      = self.infected_radius / 2
            self.mass             = self._nominal_mass
            self.thrust_power     = self._nominal_thrust_power

        elif self.dr_mode == "full":
            # Randomise all physical parameters
            self.wind_mag         = float(np.random.uniform(0.0, 1.0))
            self.wind_dir         = float(np.degrees(np.random.uniform(0.0, 2 * np.pi)))
            self.action_noise_std = float(np.random.uniform(0.01, 0.10))
            r0 = self._nominal_infected_radius
            self.infected_radius  = float(np.random.uniform(0.8 * r0, 1.2 * r0))
            self.spray_sigma      = self.infected_radius / 2
            self.mass             = float(np.random.uniform(0.9, 1.1))
            self.thrust_power     = 0.5 * float(np.random.uniform(0.8, 1.2))

        # ── Randomise starting positions ─────────────────────────────
        self.robot_positions = self.init_robot_positions + np.random.normal(
            0, self.init_position_noise, self.init_robot_positions.shape)

        # ── Initialise dynamic state ─────────────────────────────────
        self.robot_velocities = np.zeros((self.num_robots, 2), dtype=np.float32)
        self.robot_capacities = self.init_robot_capacities.copy()
        self.infected_positions = self.init_infected_positions.copy()
        self.infected_levels    = self.init_infected_levels.copy()

        # ── Episode counters ─────────────────────────────────────────
        self.step_count       = 0
        self.total_path_length = 0.0
        self.prev_positions   = self.robot_positions.copy()
        self.trajectories     = [[] for _ in range(self.num_robots)] # Only for rendering

        return self._get_obs()

    # ── step ─────────────────────────────────────────────────────────
    def step(self, actions):
        self.step_count += 1
        reward = 0
        terminated, truncated = False, False

        # ── Stochastic wind for this step ────────────────────────────
        wind_mag = self.wind_mag + np.random.normal(0, self.wind_noise_std)
        wind_dir = self.wind_dir + np.random.normal(0, self.wind_dir_noise_std)
        theta = np.radians(wind_dir)
        wind  = np.array([wind_mag * np.cos(theta), wind_mag * np.sin(theta)])

        total_sprayed = 0                       # Track total infection reduced this step (for reward)
        for i in range(self.num_robots):        # For each robot
            ax, ay, spray = actions[i]          # Get actual actions: thrust_x, thrust_y, sigma (spray amount)

            # Action noise + scaling
            ax = ax * self.thrust_power + np.random.normal(0, self.action_noise_std)
            ay = ay * self.thrust_power + np.random.normal(0, self.action_noise_std)

            # Velocity update (Newtonian dynamics)
            self.robot_velocities[i] += np.array([ax, ay]) / self.mass + wind
            self.robot_velocities[i]  = np.clip(
                self.robot_velocities[i], self.min_speed, self.max_speed)

            # Position update
            new_pos = self.robot_positions[i] + self.robot_velocities[i] # Predict next position
            outside = not is_inside_polygon(new_pos, self.poly_vertices) # Check if new position is outside the field
            if outside:                                 # If new position is outside the field
                reward -= 50                            # Penalty for moving outside the field
                self.robot_velocities[i] = 0            # Set all robot velocities to zero
            else:
                self.robot_positions[i] = new_pos       # Else, move to the new location

            # Spray dynamics (vectorized + capacity constraint)
            if spray > 0 and self.robot_capacities[i] > 0:                              # Only spray if spray rate > 0 and remaining spraying capacities > 0
                dists = np.linalg.norm(
                    self.robot_positions[i] - self.infected_positions, axis=1)          # Distances from robot to infected locations
                mask  = (dists <= self.infected_radius) & (self.infected_levels > 0)    # Only affect nearby infected locations

                if np.any(mask):                                                                # If nearby infected locations exists
                    w = np.exp(-(dists[mask] ** 2) / (2 * self.spray_sigma ** 2))               # Gaussian decay: closer points get more spray effect
                    spray_effect = spray * w                                                    # Scale spray intensity by distance weighting
                    spray_effect += np.random.normal(0, self.spray_noise_std, size=w.shape)     # Add stochasticity to spraying process
                    spray_effect  = np.clip(spray_effect, 0, None)                              # Prevent negative spray
                    applied       = np.minimum(spray_effect, self.infected_levels[mask])        # Cannot disinfect more infection than exists
                    total         = np.sum(applied)                                             # Total spray to apply
                    if total > self.robot_capacities[i]:                        # If exceeding available spray capacity
                        applied *= self.robot_capacities[i] / (total + 1e-8)    # Scale down proportionally
                        total    = self.robot_capacities[i]                     # Total spray applied by this robot in this step
                    self.robot_capacities[i]    -= total                        # Reduce sprayed amount from capacity
                    self.infected_levels[mask]  -= applied                      # Reduce infection levels by applied spray amount
                    total_sprayed               += total                        # Accumulate global spraying amount for all robots
                else:
                    # Useless spray penalty (part of R_eff)
                    if self.reward_ablation != "no_spr":
                        reward -= 0.05 * spray

            # ── Per-robot movement penalties ─────────────────
            if self.reward_ablation != "no_path":
                reward -= 0.1 * (ax ** 2 + ay ** 2)                       # Energy penalty (encourages efficient control)
                reward -= 0.3 * np.linalg.norm(self.robot_velocities[i])  # Movement penalty (Penalize high speeds)

            # Trajectory buffer (rendering only)
            self.trajectories[i].append(self.robot_positions[i].copy())
            if len(self.trajectories[i]) > 200:
                self.trajectories[i].pop(0)

        # Path length computation (true traveled distance)
        step_dist = np.linalg.norm(self.robot_positions - self.prev_positions, axis=1)   # Distance traveled by each robot this step
        step_path = np.sum(step_dist)                                                    # Total distance traveled by all robots
        self.total_path_length += step_path                                              # Accumulate episode path length
        self.prev_positions = self.robot_positions.copy()                                # Update previous positions

        # ── R_spray: core spraying reward ────────────────────────────
        if self.reward_ablation != "no_spr":
            reward += 100.0 * total_sprayed     # Strong positive reward for removing infection

        # ── R_eff: global efficiency penalties ───────────────────────
        if self.reward_ablation != "no_path":
            reward -= 1.0 * step_path       # path length penalty
            reward -= 2.0                   # time penalty

        # Remaining infection penalty
        remaining = np.sum(self.infected_levels)
        if self.reward_ablation != "no_spr":
            # Distance shaping to nearest active infection
            active_mask = self.infected_levels > 0.01
            if np.any(active_mask):
                active_pos = self.infected_positions[active_mask]
                dists_mat  = np.linalg.norm(
                    self.robot_positions[:, None] - active_pos[None, :], axis=2)
                nearest    = np.min(dists_mat, axis=1)
                reward    += 0.5 * np.sum(np.exp(-nearest))            
            reward -= 0.5 * remaining

        term_cond = ""               # Terminal conditions for the reward ablation
        if remaining <= 0.01:
            if self.reward_ablation != "no_term":
                reward += 500.0      # Large positive reward for successfull spraying
            term_cond = "sprayed"   
            terminated = True        # always terminate on succession (completion)

        # ── R_col: collision penalty ──────────────────────────────────
        if self.num_robots > 1:                                             # Check collision only for multi-robots
            if compute_min_dist(self.robot_positions) < self.robot_size:    # Robots too close
                if self.reward_ablation != "no_term":
                    reward -= 1000.0                                        # Large negative reward for collision
                term_cond = "collision"
                terminated = True   # always terminate on collision (safety)

        # ── Build observation & info ──────────────────────────────────
        obs, info = self._get_obs()
        truncated = self.step_count >= self.max_steps       # Episode ends if max steps reached
        if truncated:
            term_cond = "max_steps"
        info.update({
            "total_sprayed": float(total_sprayed),          # Total infection removed this step
            "remaining_infection": float(remaining),        # Remaining infected amount
            "episode_length": self.step_count,              # Current timestep
            "path_length": float(self.total_path_length),   # Total distance traveled
            "term_cond": term_cond
        })

        return obs, reward, terminated, truncated, info

    # ── render ───────────────────────────────────────────────────────
    def render(self):
        if self.render_mode != "human": 
            return              # Only render if mode is set to "human" (skip rendering for training efficiency)

        if self.screen is None:
            pygame.init()                                                                   # Initialize all pygame modules
            pygame.display.init()                                                           # Initialize display module explicitly
            self.screen = pygame.display.set_mode((self.screen_width, self.screen_height))  # Create window with predefined screen size
            pygame.display.set_caption("Multi-Robot Spraying Environment")                  # Window title
            self.clock = pygame.time.Clock()                                                # Clock to control rendering FPS

        self.screen.fill((255, 255, 255))       # Fill screen with white color

        # --- Draw field boundary ---
        scaled_poly = [self.world_to_screen(p) for p in self.poly_vertices] # Convert field vertices from world coordinates → screen coordinates
        pygame.draw.polygon(self.screen, (255, 255, 0), scaled_poly)

        # --- Draw robot trajectories (paths) ---
        for i in range(self.num_robots):
            if len(self.trajectories[i]) > 1:                                       # Only draw if at least two points exist (needed for a line)
                points = [self.world_to_screen(p) for p in self.trajectories[i]]    # Convert trajectory points to screen coordinates
                pygame.draw.lines(self.screen, (150, 150, 150), False, points, 2)   # Draw polyline showing path history

        # --- Draw robots (current positions) ---        
        for i in range(self.num_robots):
            pos = self.world_to_screen(self.robot_positions[i])                 # Convert robot position to screen coordinates
            pygame.draw.circle(self.screen,
                               self.robot_colors[i % len(self.robot_colors)],   # Assign color cyclically
                               pos, self.robot_size * self.render_scale / 2)    # Scale radius for visualization

        # --- Draw infected locations (dynamic color) ---
        for pos, level in zip(self.infected_positions, self.infected_levels):
            intensity = int(255 * min(level / 5.0, 1.0))                          # Map infection level → color intensity (visual feedback)
            pygame.draw.circle(self.screen, (0, 255-intensity, 255),              # Higher infection → darker blue and lower infection → lighter blue
                               self.world_to_screen(pos), self.render_scale + 2)  # Fixed radius for visibility
        
        # Update display
        pygame.display.flip()  # Push all drawings to the screen (double buffering)
        self.clock.tick(60)    # Limit rendering to 60 FPS for smooth visualization

    def close(self):
        if self.screen:
            pygame.quit()

In [ ]:
# ================================
# Register and load the configuation files
# ================================
num_robots = 3
height = 0.4
gym.register(id='MultiRobotEnv-v0', entry_point=MultiRobotEnv, max_episode_steps=1000)
json_path = os.path.join('..', 'exp_sets', 'stochastic_envs_v2.json')
json_dict = load_experiment_dict_json(json_path)

In [ ]:
# ================================
# Initialize the ZeroMQ Client
# ================================
client = RemoteAPIClient()
sim = client.getObject('sim')
defaultIdleFps = sim.getInt32Param(sim.intparam_idle_fps)
sim.setInt32Param(sim.intparam_idle_fps, 0)

# ================================
# Drone simulator class
# ================================
class Drone_simulator:
    def __init__(self, gym_env, scaling_factor=5, height=1, num_robots=3):
        self.scaling_factor = scaling_factor
        self.polygon = gym_env.unwrapped.poly_vertices
        self.scaled_polygon = [(x/scaling_factor, y/scaling_factor) for (x,y) in self.polygon]
        self.rounded_polygon = self.scaled_polygon + [self.scaled_polygon[0]]
        self.weed_locations=[loc for loc in gym_env.unwrapped.init_infected_positions]
        self.height = height
        self.num_robots = num_robots
        self.max_robots = 5

        # cache script handles (important!)        
        self.nozzle_scripts = []
        for i in range(num_robots):
            # Directly get the Lua script handle
            script_path = f"/Quadcopter[{i}]/PaintNozzle/Script"
            script_handle = sim.getObject(script_path)
            self.nozzle_scripts.append(script_handle)
        
        # Drawing handle
        self.field_drawing = None

        # Track spawned weeds (IMPORTANT)
        self.spawned_weeds = []

        # ---- cache quadcopters & initial states ----
        self.all_drones = []
        self.initial_positions = {}
        i = 0
        while True:
            h = sim.getObject(f"/Quadcopter[{i}]", {'noError': True})
            if h == -1:
                break
            self.all_drones.append(h)
            self.initial_positions[h] = sim.getObjectPosition(h, -1)
            i += 1

    # ------------------------------------------------

    def start_simulation(self):
        sim.startSimulation()
        print("Simulation started")

    def stop_simulation(self):
        sim.stopSimulation()
        while sim.getSimulationState() != sim.simulation_stopped:
            sim.step()
        
        # ---- cleanup runtime artifacts ----
        self.stop_spraying()
        self.clear_field()
        self.clear_weeds()

        # ---- restore robots ----
        for drone, pos in self.initial_positions.items():
            sim.setObjectPosition(drone, -1, pos)
            sim.setModelProperty(drone, 0)
        print("Simulation reset completed")

    # Field drawing
    def draw_field(self):
        white = [255, 255, 255]
        self.field_drawing = sim.addDrawingObject(sim.drawing_lines, 5, 0, -1, 9999, white)

        for i in range(len(self.rounded_polygon) - 1):
            p1 = self.rounded_polygon[i]
            p2 = self.rounded_polygon[i+1]
            line = [
                p1[0], p1[1], 0.1,
                p2[0], p2[1], 0.1     # drawing at height 0.1
            ]
            sim.addDrawingObjectItem(self.field_drawing, line)
    
    def clear_field(self):
        if self.field_drawing is not None:
            sim.removeDrawingObject(self.field_drawing)
            self.field_drawing = None

    # ------------------------------------------------

    def set_agent_positions(self, info):
        for i, drone in enumerate(self.all_drones):
            if i < self.num_robots:
                pos = info[f'robot{i}']['position']
                pos = [p / self.scaling_factor for p in pos] + [self.height]
                sim.setObjectPosition(drone, -1, pos)
                sim.setObjectInt32Param(
                    drone, sim.objintparam_visibility_layer, 1
                )
            else:
                # hide unused drones
                sim.setModelProperty(
                    drone,
                    sim.modelproperty_not_visible
                    | sim.modelproperty_not_collidable
                    | sim.modelproperty_not_detectable
                    | sim.modelproperty_not_dynamic
                )
    
    def set_weed_locations(self):
        weed_template = sim.getObject('/weed')

        # ---- clear previous weeds ----
        self.clear_weeds()

        # ---- spawn new weeds ----
        for loc in self.weed_locations:
            x = [xi / self.scaling_factor for xi in loc]
            new_pos = x + [0]

            new_weed = sim.copyPasteObjects([weed_template])[0]
            sim.setObjectPosition(new_weed, -1, new_pos)

            self.spawned_weeds.append(new_weed)
    
    def clear_weeds(self):
        for obj in self.spawned_weeds:
            if sim.isHandle(obj):
                sim.removeObject(obj)
        self.spawned_weeds = []

    # ------------------------------------------------
    def move_agents(self, info, action):
        """
        action[i] = [dx, dy, spray]
        spray in [0,1]
        """
        for i in range(self.num_robots):
            # ---- move target (position control) ----
            target = sim.getObject(f"/target[{i}]")
            pos = info[f'robot{i}']['position']
            # print("position", pos)
            pos = [p/self.scaling_factor for p in pos] + [self.height]
            sim.setObjectPosition(target, -1, pos)

            # ---- spray control via Lua ----
            spray = float(action[i][2])
            enable = spray > 0.01

            sim.callScriptFunction(
                "setSprayCommand",
                self.nozzle_scripts[i],
                enable,
                spray
            )
    def stop_spraying(self):
        for i in range(self.num_robots):
            sim.callScriptFunction(
                    "setSprayCommand",
                    self.nozzle_scripts[i],
                    True,
                    0
                )

# ================================
# Function to run simulation
# ================================
def run_simulation(env_id, trained_model_path):
    model = CrossQ.load(trained_model_path)

    # Create Gym environment
    env = gym.make('MultiRobotEnv-v0', render_mode='human', field_info=json_dict[f"set{env_id}"], num_robots=num_robots, max_steps=1000)
    env.metadata['render_fps'] = 1
    obs, info = env.reset()
    env.render()

    # Create simulator
    # Make the simulator object, draw the field, and set agent positions
    drone_simulator = Drone_simulator(gym_env=env, scaling_factor=8, height=height, num_robots=num_robots)
    # drone_simulator.start_simulation()
    drone_simulator.draw_field()
    drone_simulator.set_agent_positions(info=info)
    drone_simulator.set_weed_locations()
    print(info)

    input("Press any key to start simulation:")

    # Start CoppeliaSim
    drone_simulator.start_simulation()
    terminated = False
    truncated = False
    total_rewards = 0
    while True:
        action, _ = model.predict(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        env.render()
        total_rewards += reward
        print(
            f"reward={reward:.2f}, "
            f"total={total_rewards:.2f}, "
            f"spray={action[:,2]}"
        )
        drone_simulator.move_agents(info, action)
        if terminated or truncated:
            print("Episode finished")        
            drone_simulator.stop_spraying()
            break
        pygame.event.get()
        pygame.time.wait(200)
    
    return env, drone_simulator

In [ ]:
assert False, "Please open the scene file '.\simulation\sim_envs\coppeliasim_scene_for_spraying_v3.ttt' before running the following cells"

### Run simulation on the 10 environment variations

#### Environment 1

In [ ]:
# Load trained network
trained_model_path = r'..\logs\Apr20_21_v1_seed123\env1_CrossQ.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=1, trained_model_path=trained_model_path)

Stop simulation (IMPORTANT! otherwise the scene file will be changed if accidentally saved.)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 2

In [ ]:
# Load trained network
trained_model_path = r'..\logs\Apr20_21_v1_seed123\env2_CrossQ.zip' # similar to best model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env2\best_model.zip' # Good model

# Start simulation!
env, drone_simulator = run_simulation(env_id=2, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 3

In [ ]:
# Load trained network
trained_model_path = r'..\logs\Apr20_21_v1_seed123\env3_CrossQ.zip' # Best model is better
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env3\best_model.zip' # Good

# Start simulation!
env, drone_simulator = run_simulation(env_id=3, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 4

In [ ]:
# Load trained network
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\env4_CrossQ.zip'
trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env4\best_model.zip'

# Start simulation!
env, drone_simulator = run_simulation(env_id=4, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 5

In [ ]:
# Load trained network
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\env5_CrossQ.zip'
trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env5\best_model.zip'

# Start simulation!
env, drone_simulator = run_simulation(env_id=5, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 6

In [ ]:
# Load trained network
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\env6_CrossQ.zip'
trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env6\best_model.zip'

# Start simulation!
env, drone_simulator = run_simulation(env_id=6, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 7

In [ ]:
# Load trained network
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\env7_CrossQ.zip'
trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env7\best_model.zip'

# Start simulation!
env, drone_simulator = run_simulation(env_id=7, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 8

In [ ]:
# Load trained network
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\env8_CrossQ.zip'
trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env8\best_model.zip'

# Start simulation!
env, drone_simulator = run_simulation(env_id=8, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 9

In [ ]:
# Load trained network
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\env9_CrossQ.zip'
trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env9\best_model.zip'

# Start simulation!
env, drone_simulator = run_simulation(env_id=9, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 10

In [ ]:
# Load trained network
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\env10_CrossQ.zip'
trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env10\best_model.zip'

# Start simulation!
env, drone_simulator = run_simulation(env_id=10, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

**IMPORTANT**: When closing the CoppeliaSim scene file, please select "No" for the question "Do you wish to save the changes?"